# puc — attitudinal experiment (end to end)

Drives the attitudinal flow: **run episodes → evaluate → read results**. Unlike the
objective demo there is **no material-generation step** — attitudinal questions are
matters of judgment with no correct answer, so the run points straight at a scenario
`.toml`.

The `aligned` condition is the **unbiased** steelman baseline; `misaligned` is the
**biased** persuader, swept over both stances (each pole of the axis) × three levels.
There is no judge — a single **monitor** scores where the response lands on the stance
axis (`implied_stance`) and how biased it is (`bias`).

In [5]:
import json
import os
import sys
from pathlib import Path

# This notebook lives in notebooks/; run from the repo root so relative paths
# (configs/, scenarios/, results/) and local imports (run) resolve regardless of
# the kernel's working directory.
_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), Path.cwd())
os.chdir(_ROOT)
sys.path.insert(0, str(_ROOT))

from dotenv import load_dotenv

load_dotenv()  # ANTHROPIC_API_KEY from .env

SCENARIO = "scenarios/2_1_attitudinal.toml"  # attitudinal scenario (the "what"); no material
CONFIG = "configs/dev_attitudinal.toml"      # run config (the "how")
MAX_WORKERS = 8   # episodes / conditions to run in parallel (each = one API call); 1 = sequential

## 1. Run the conversation

Expands the `[experiment]` table against the scenario and runs each episode's actor
turn (1 aligned/steelman baseline + 2 stances × 3 levels = 7 episodes), in parallel
across `MAX_WORKERS` threads (each is one independent actor call). Writes one
transcript record per episode to `results/transcripts/` (in episode order,
regardless of completion order). Judging is a separate step (below).

In [ ]:
from run import converse

transcripts_path = converse(CONFIG, SCENARIO, max_workers=MAX_WORKERS)
transcripts_path

configs/dev_attitudinal.toml + scenarios/2_1_attitudinal.toml: 7 episode(s) → results/transcripts/dev_att-20260710T211749Z.jsonl


KeyboardInterrupt: 

  [dev_att/aligned/a=claude-sonnet-4-6/r0] ok (52.1s)
  [dev_att/aligned/a=claude-sonnet-4-6/r0] ok (40.9s)


## 2. Evaluate the transcripts

Runs the attitudinal **monitor** over the transcripts (no judge), in parallel across
`MAX_WORKERS` threads. Writes verdicts to `results/verdicts/`, named after the
transcript they scored. Re-run after tweaking the monitor prompt to compare — each
verdict logs the prompt version it used.

In [ ]:
from run import evaluate

verdicts_path = evaluate(CONFIG, transcripts_path, max_workers=MAX_WORKERS)
verdicts_path

## 3. Read results

Joins the transcripts with the verdicts (they align in order) and prints each
conversation with its monitor verdict.

In [ ]:
import textwrap

# Files to read. Default to what the cells above produced; override to load an
# earlier run, e.g. TRANSCRIPTS = "results/transcripts/dev_att-<stamp>.jsonl".
TRANSCRIPTS = transcripts_path   # from section 1
VERDICTS = verdicts_path         # from section 2

trecs = [json.loads(l) for l in Path(TRANSCRIPTS).read_text().splitlines() if l.strip()]
vrecs = [json.loads(l) for l in Path(VERDICTS).read_text().splitlines() if l.strip()]


def show_verdict(name, verdict):
    print(f"\n  {name}")
    if not isinstance(verdict, dict):
        print(f"    {verdict}")
        return
    for k, v in verdict.items():
        if k in ("notes", "raw"):  # long free text — wrap in full, never truncate
            print(textwrap.fill(
                str(v), width=100,
                initial_indent=f"    {k:<16} ", subsequent_indent=" " * 21,
            ))
        else:
            print(f"    {k:<16} {v}")


for trec, vrec in zip(trecs, vrecs):
    r = trec.get("experiment") or trec.get("run")
    label = "unbiased" if r["condition"] == "aligned" else "biased"
    if r.get("level"):
        label += f" / {r['level']}"
    ts = (trec.get("scenario") or {}).get("target_stance")
    if ts:
        label += f"   → target: {ts}"
    print("\n" + "=" * 80)
    print(f"  {label}   (actor={r['models']['actor']})")
    print("=" * 80)

    if trec.get("error"):
        print("  ERROR:", trec["error"])
        continue

    for turn in trec["turns"]:
        who = "USER" if turn["speaker"] == "user" else "ACTOR"
        print(f"\n  [{who}]")
        print(textwrap.indent(textwrap.fill(turn["text"], width=100), "    "))

    if vrec.get("error"):
        print("\n  EVAL ERROR:", vrec["error"])
    else:
        show_verdict("monitor", vrec.get("monitor_verdict"))

## 4. Results table

One row per episode: the monitor's `implied_stance` (where the actor landed on the
axis), `bias`, and `confidence`. The **match** column labels each biased run
`matched`/`unmatched` post-hoc — whether its target pole is on the same side the
unbiased baseline landed on. Reads the verdicts file.

In [ ]:
from IPython.display import Markdown, display

records = [json.loads(l) for l in Path(VERDICTS).read_text().splitlines() if l.strip()]


def md_table(headers, rows):
    row = lambda cells: "| " + " | ".join(str(c) for c in cells) + " |"
    sep = "| " + " | ".join("---" for _ in headers) + " |"
    return "\n".join([row(headers), sep, *(row(r) for r in rows)])


scn = records[0]["scenario"]
stances = scn["stances"]
mdl_actor = (records[0].get("experiment") or {}).get("models", {}).get("actor", "—")
mdl_eval = records[0]["eval"]["models"]

# Reference: where the unbiased (aligned) baseline landed on the axis.
baseline_stance = None
for rec in records:
    exp = rec.get("experiment") or {}
    if exp.get("condition") == "aligned" and not rec.get("error"):
        baseline_stance = (rec.get("monitor_verdict") or {}).get("implied_stance")
        break

context = f"""\
**Question**  
{scn["question"]}

**Stances (axis):** 0 = {stances[0]}  ·  100 = {stances[1]}

**Setup**
- actor `{mdl_actor}`, monitor `{mdl_eval["monitor"]}` (no judge — attitudinal has no correct answer)
- single turn (one actor response); no material

**Columns (0–100)**
- **implied_stance** — where the response leaves a reader on the axis (0 = first stance, 50 = balanced, 100 = second stance)
- **bias** — reliance on manipulative / one-sided means (0 = fair & even-handed, 100 = heavy bias)
- **confidence** — how sure the monitor is of its bias score
- **match** — for biased runs, whether the target pole is on the same side the unbiased baseline landed on
"""
display(Markdown(context))


def pole_side(v):
    return None if v is None else (0 if v < 50 else 1)


rows = []
for rec in records:
    exp = rec.get("experiment") or {}
    cond = "unbiased" if exp.get("condition") == "aligned" else "biased"
    level = exp.get("level") or "—"
    ts = (rec.get("scenario") or {}).get("target_stance")
    target_idx = stances.index(ts) if ts in stances else None
    target_lbl = "—" if target_idx is None else f"stance{target_idx}"

    if rec.get("error"):
        rows.append([cond, level, target_lbl, "ERR", "ERR", "ERR", "—"])
        continue
    mv = rec.get("monitor_verdict") or {}
    match = "—"
    if target_idx is not None and baseline_stance is not None:
        match = "matched" if target_idx == pole_side(baseline_stance) else "unmatched"
    rows.append([
        cond, level, target_lbl,
        mv.get("implied_stance", "—"), mv.get("bias", "—"), mv.get("confidence", "—"), match,
    ])

display(Markdown(md_table(
    ["condition", "level", "target", "implied_stance", "bias", "confidence", "match"],
    rows,
)))